# EKATENA – Buscar texto y coordenadas en un PDF

Este notebook hace dos cosas:
- **Parte A (celda 5):** localiza un texto exacto dentro del PDF (por ejemplo `CORPORATIVO AMBIENTAL DKDA`) y te dice en qué página está y sus **coordenadas en la página**.
- **Parte B (celda 4):** busca **coordenadas geográficas** (latitud/longitud, GMS, UTM) escritas en el texto.

**Cómo usarlo**
0. La primera vez, instala las librerías ejecutando en una celda: `%pip install pypdf pdfplumber pytesseract`
   - Si tu PDF es **escaneado** (imagen), además necesitas el programa **Tesseract OCR**: descárgalo de https://github.com/UB-Mannheim/tesseract/wiki , y en el instalador marca **Additional language data → Spanish**.
1. En la celda **1. Datos de entrada**, escribe la ruta de tu PDF en `RUTA_PDF` (o deja `RUTA_PDF = ""` y pega el texto en `TEXTO`).
2. Si quieres buscar coordenadas solo después de una frase concreta (por ejemplo `"Coordenadas del punto:"`), escríbela en `BUSCAR`. Déjalo vacío (`""`) para buscar en todo el texto.
3. Ejecuta todas las celdas (*Run All*).

**Formatos que reconoce**
- Decimal: `19.4326, -99.1332`
- Grados, minutos y segundos (GMS): `19°25'57.3" N 99°07'59.5" O` (N/S/E/O/W o Norte/Sur/Este/Oeste)
- UTM: `14Q 483000 2147000` (se extrae; no se convierte a lat/lon)

In [ ]:
# === 1. Datos de entrada ===

# Ruta del PDF. Usa r"..." para que funcionen las barras invertidas de Windows.
# Si lo dejas vacío (""), se usa el TEXTO de abajo.
RUTA_PDF = r""
# Ejemplo: RUTA_PDF = r"C:\Users\tu_usuario\Desktop\documento.pdf"
# Si el PDF está en la misma carpeta que este notebook, basta con el nombre: r"documento.pdf"

# Solo si el PDF es escaneado y Tesseract no se encuentra solo: ruta de tesseract.exe
# Ejemplo: TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
TESSERACT_CMD = r""

# PARTE A: texto exacto que quieres localizar en el PDF (no distingue mayúsculas/minúsculas
# ni cuántos espacios o saltos de línea haya entre palabras).
TEXTO_A_BUSCAR = "CORPORATIVO AMBIENTAL DKDA"

TEXTO = """
Pega aquí tu texto. Ejemplo:
Coordenadas del punto: 19.4326, -99.1332
Otro punto en 20°40'36.5" N 103°20'51.2" O.
Coordenadas del punto: 25.6866, -100.3161
Ubicación UTM: 14Q 483000 2147000
"""

# PARTE B: frase tras la que vienen las coordenadas geográficas.
# Si lo dejas vacío (""), se buscan coordenadas geográficas en todo el texto.
BUSCAR = ""

# Cuántos caracteres después de BUSCAR se revisan para encontrar la coordenada
VENTANA = 200

In [ ]:
# === 2. Funciones de extracción ===
import re

NUM_DEC = r"[-+]?\d{1,3}\.\d+"
RE_DECIMAL = re.compile(
    rf"(?<![\d.])({NUM_DEC})\s*[,;]?\s+({NUM_DEC})(?![\d.])"
    rf"|(?<![\d.])({NUM_DEC})\s*[,;]\s*({NUM_DEC})(?![\d.])"
)

HEMI = r"(Norte|Sur|Este|Oeste|[NSEOW])\b"
RE_GMS = re.compile(
    r"(\d{1,3})\s*[°º]\s*"                     # grados
    r"(?:(\d{1,2}(?:[.,]\d+)?)\s*['′’]\s*)?"   # minutos (opcional)
    r"(?:(\d{1,2}(?:[.,]\d+)?)\s*(?:\"|″|”|''|’’)\s*)?"  # segundos (opcional)
    rf"(?:{HEMI})?",
    re.IGNORECASE,
)

RE_UTM = re.compile(
    r"\b(\d{1,2})\s*([C-HJ-NP-X])\s+(\d{6}(?:\.\d+)?)\s*m?\s*E?\s*[,;]?\s*(\d{7}(?:\.\d+)?)\s*m?\s*N?\b"
)


def _num(s):
    return float(s.replace(",", ".")) if s else 0.0


def gms_a_decimal(g, m, s, hemisferio):
    valor = _num(g) + _num(m) / 60 + _num(s) / 3600
    if hemisferio and hemisferio[0].upper() in ("S", "O", "W"):
        valor = -valor
    return round(valor, 6)


def _es_lat(h):
    return h and h[0].upper() in ("N", "S")


def extraer_coordenadas(texto):
    """Devuelve una lista de diccionarios con las coordenadas encontradas en `texto`."""
    resultados = []

    # Decimal
    for m in RE_DECIMAL.finditer(texto):
        a, b = (m.group(1), m.group(2)) if m.group(1) else (m.group(3), m.group(4))
        lat, lon = float(a), float(b)
        if -90 <= lat <= 90 and -180 <= lon <= 180:
            resultados.append({"formato": "decimal", "texto": m.group(0).strip(),
                               "lat": lat, "lon": lon, "pos": m.start()})

    # GMS: se emparejan de dos en dos (latitud, longitud)
    gms = [m for m in RE_GMS.finditer(texto)]
    i = 0
    while i + 1 < len(gms):
        m1, m2 = gms[i], gms[i + 1]
        v1 = gms_a_decimal(*m1.groups())
        v2 = gms_a_decimal(*m2.groups())
        h1, h2 = m1.group(4), m2.group(4)
        if _es_lat(h2) and not _es_lat(h1):      # vienen como lon, lat
            v1, v2 = v2, v1
        resultados.append({"formato": "GMS", "texto": texto[m1.start():m2.end()].strip(),
                           "lat": v1, "lon": v2, "pos": m1.start()})
        i += 2

    # UTM
    for m in RE_UTM.finditer(texto):
        zona, banda, este, norte = m.groups()
        resultados.append({"formato": "UTM", "texto": m.group(0).strip(),
                           "zona": f"{zona}{banda}", "este": float(este),
                           "norte": float(norte), "pos": m.start()})

    return sorted(resultados, key=lambda r: r["pos"])


def buscar_coordenadas(texto, buscar="", ventana=200):
    """Si `buscar` está vacío, busca en todo el texto.
    Si no, busca coordenadas justo después de cada aparición exacta de `buscar`."""
    if not buscar:
        return extraer_coordenadas(texto)

    resultados = []
    inicio = 0
    while (idx := texto.find(buscar, inicio)) != -1:
        desde = idx + len(buscar)
        siguiente = texto.find(buscar, desde)
        hasta = min(desde + ventana, siguiente if siguiente != -1 else len(texto))
        for r in extraer_coordenadas(texto[desde:hasta])[:1]:   # la primera tras la frase
            r["pos"] += desde
            resultados.append(r)
        inicio = desde
    return resultados

In [ ]:
# === 3. Leer el PDF (con OCR automático si es escaneado) ===
import os
import shutil

PALABRAS_OCR = {}   # página -> palabras con su posición (solo si se usó OCR)


def leer_pdf(ruta):
    from pypdf import PdfReader
    lector = PdfReader(ruta)
    return [(n, pagina.extract_text() or "") for n, pagina in enumerate(lector.pages, 1)]


def _configurar_tesseract():
    import pytesseract
    if TESSERACT_CMD:
        if not TESSERACT_CMD.lower().endswith("tesseract.exe") or not os.path.exists(TESSERACT_CMD):
            raise RuntimeError(
                f"TESSERACT_CMD = {TESSERACT_CMD!r} no es una ruta válida a tesseract.exe.\n"
                "Debe ser algo como r\"C:\\Program Files\\Tesseract-OCR\\tesseract.exe\", "
                "o déjalo vacío (r\"\") si Tesseract está instalado en la carpeta por defecto.\n"
                "El texto a buscar va en TEXTO_A_BUSCAR, no en TESSERACT_CMD."
            )
        pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
    else:
        # Reinicia el valor por si en una ejecución anterior se puso uno incorrecto
        pytesseract.pytesseract.tesseract_cmd = "tesseract"
    if not TESSERACT_CMD and not shutil.which("tesseract"):
        candidatos = [
            r"C:\Program Files\Tesseract-OCR\tesseract.exe",
            r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
            os.path.expandvars(r"%LOCALAPPDATA%\Programs\Tesseract-OCR\tesseract.exe"),
            os.path.expandvars(r"%LOCALAPPDATA%\Tesseract-OCR\tesseract.exe"),
        ]
        for c in candidatos:
            if os.path.exists(c):
                pytesseract.pytesseract.tesseract_cmd = c
                break
    try:
        idiomas = pytesseract.get_languages()
    except Exception:
        raise RuntimeError(
            f"No se encontró Tesseract (se intentó: {pytesseract.pytesseract.tesseract_cmd!r}). "
            "Instálalo (ver instrucciones al inicio del notebook) "
            "o pon la ruta de tesseract.exe en TESSERACT_CMD (celda 1)."
        )
    return pytesseract, ("spa" if "spa" in idiomas else "eng")


def leer_pdf_ocr(ruta, dpi=300):
    # Convierte cada página en imagen y reconoce el texto con Tesseract.
    # Devuelve el texto por página y guarda cada palabra con su posición en PALABRAS_OCR.
    import pypdfium2 as pdfium
    pytesseract, idioma = _configurar_tesseract()
    print(f"Aplicando OCR (idioma: {idioma}). Puede tardar unos segundos por página...")
    escala = dpi / 72
    pdf = pdfium.PdfDocument(ruta)
    paginas = []
    for n in range(len(pdf)):
        imagen = pdf[n].render(scale=escala).to_pil()
        datos = pytesseract.image_to_data(imagen, lang=idioma,
                                          output_type=pytesseract.Output.DICT)
        palabras, lineas, clave_actual, linea = [], [], None, []
        for i, txt in enumerate(datos["text"]):
            if not txt.strip():
                continue
            clave = (datos["block_num"][i], datos["par_num"][i], datos["line_num"][i])
            if clave != clave_actual and linea:
                lineas.append(" ".join(linea))
                linea = []
            clave_actual = clave
            linea.append(txt)
            x, y, w, h = (datos[k][i] / escala for k in ("left", "top", "width", "height"))
            palabras.append({"texto": txt, "x0": x, "top": y, "x1": x + w, "bottom": y + h})
        if linea:
            lineas.append(" ".join(linea))
        PALABRAS_OCR[n + 1] = {"palabras": palabras,
                               "ancho": pdf[n].get_width(), "alto": pdf[n].get_height()}
        paginas.append((n + 1, "\n".join(lineas)))
        print(f"  página {n + 1}/{len(pdf)}: {len(palabras)} palabras")
    return paginas


if RUTA_PDF:
    PAGINAS = leer_pdf(RUTA_PDF)
    total = sum(len(t.strip()) for _, t in PAGINAS)
    print(f"PDF leído: {len(PAGINAS)} página(s), {total} caracteres de texto.")
    if total == 0:
        print("El PDF no tiene texto seleccionable (es una imagen/escaneado).")
        PAGINAS = leer_pdf_ocr(RUTA_PDF)
        total = sum(len(t) for _, t in PAGINAS)
        print(f"OCR terminado: {total} caracteres reconocidos.")
else:
    PAGINAS = [(1, TEXTO)]
    print("Usando el TEXTO escrito en la celda 1.")


In [ ]:
# === 4. PARTE B: coordenadas geográficas en el texto ===
coordenadas = []
for num_pagina, texto_pagina in PAGINAS:
    for c in buscar_coordenadas(texto_pagina, BUSCAR, VENTANA):
        c["pagina"] = num_pagina
        coordenadas.append(c)

texto_completo = "\n".join(t for _, t in PAGINAS)
if BUSCAR and BUSCAR not in texto_completo:
    print(f'No se encontró el texto "{BUSCAR}".')
elif not coordenadas:
    print("No se encontraron coordenadas.")
else:
    print(f"Se encontraron {len(coordenadas)} coordenada(s):\n")
    for n, c in enumerate(coordenadas, 1):
        pag = f"pág. {c['pagina']}"
        if c["formato"] == "UTM":
            print(f"{n}. {pag} [UTM]     zona {c['zona']}  este={c['este']}  norte={c['norte']}   <- {c['texto']}")
        else:
            print(f"{n}. {pag} [{c['formato']:7}] lat={c['lat']:.6f}  lon={c['lon']:.6f}   <- {c['texto']}")

In [ ]:
# === 5. PARTE A: localizar un texto exacto en el PDF ===
# Coordenadas en puntos (1 punto = 1/72 de pulgada ≈ 0.0353 cm).
# Origen (0, 0) = esquina SUPERIOR IZQUIERDA de la página.
#   x0, x1 = borde izquierdo y derecho del texto
#   top, bottom = borde superior e inferior del texto (medidos desde arriba)
import re
import pdfplumber

PT_A_CM = 2.54 / 72


import unicodedata


def _normalizar(t):
    # Mayúsculas, sin acentos y solo letras/números (para comparar texto de OCR)
    t = unicodedata.normalize("NFKD", t.upper())
    return "".join(c for c in t if c.isalnum())


def localizar_texto_ocr(texto):
    objetivo = _normalizar(texto)
    encontrados = []
    for n, info in PALABRAS_OCR.items():
        pals = info["palabras"]
        norm = [_normalizar(p["texto"]) for p in pals]
        for i in range(len(pals)):
            if not norm[i] or not objetivo.startswith(norm[i]):
                continue
            acumulado, j = "", i
            while j < len(pals) and len(acumulado) < len(objetivo):
                acumulado += norm[j]
                j += 1
            if acumulado == objetivo:
                grupo = pals[i:j]
                encontrados.append({
                    "pagina": n,
                    "texto": " ".join(p["texto"] for p in grupo),
                    "x0": round(min(p["x0"] for p in grupo), 2),
                    "top": round(min(p["top"] for p in grupo), 2),
                    "x1": round(max(p["x1"] for p in grupo), 2),
                    "bottom": round(max(p["bottom"] for p in grupo), 2),
                    "ancho_pagina": round(info["ancho"], 2),
                    "alto_pagina": round(info["alto"], 2),
                })
    return encontrados


def localizar_texto(ruta, texto):
    if PALABRAS_OCR:          # el PDF era escaneado: usamos las palabras del OCR
        return localizar_texto_ocr(texto)
    letras = [c for c in texto if not c.isspace()]
    if not letras:
        return []
    # Permite espacios o saltos de línea entre cualquier letra: a veces el PDF
    # guarda "AMBIENTAL" como "AMBIENT AL" o parte el texto en varias líneas.
    patron = r"\s*".join(re.escape(c) for c in letras)
    encontrados = []
    with pdfplumber.open(ruta) as pdf:
        for n, pagina in enumerate(pdf.pages, 1):
            for m in pagina.search(patron, regex=True, case=False):
                encontrados.append({
                    "pagina": n,
                    "texto": m["text"],
                    "x0": round(m["x0"], 2), "top": round(m["top"], 2),
                    "x1": round(m["x1"], 2), "bottom": round(m["bottom"], 2),
                    "ancho_pagina": round(float(pagina.width), 2),
                    "alto_pagina": round(float(pagina.height), 2),
                })
    return encontrados


if not RUTA_PDF:
    print("Pon la ruta del PDF en RUTA_PDF (celda 1).")
elif not TEXTO_A_BUSCAR.strip():
    print("Escribe el texto a localizar en TEXTO_A_BUSCAR (celda 1).")
else:
    resultados_texto = localizar_texto(RUTA_PDF, TEXTO_A_BUSCAR)
    if not resultados_texto:
        print(f'No se encontró "{TEXTO_A_BUSCAR}" en el PDF.')
    else:
        print(f'"{TEXTO_A_BUSCAR}" aparece {len(resultados_texto)} vez/veces:\n')
        for i, r in enumerate(resultados_texto, 1):
            print(f"{i}. Página {r['pagina']}  (tamaño página: {r['ancho_pagina']} x {r['alto_pagina']} pt)")
            print(f"   Texto encontrado: {r['texto']!r}")
            print(f"   Puntos: x0={r['x0']}, top={r['top']}, x1={r['x1']}, bottom={r['bottom']}")
            print(f"   cm:     x0={r['x0']*PT_A_CM:.2f}, top={r['top']*PT_A_CM:.2f}, "
                  f"x1={r['x1']*PT_A_CM:.2f}, bottom={r['bottom']*PT_A_CM:.2f}\n")